In [ ]:
# Import navis

## Image viewing
import os
from pathlib import Path
import uuid
import math

from scipy.special import expit

from tqdm.auto import tqdm

import navis


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import confusion_matrix
import itertools

import colorcet as cc

# Import neuprint wrapper by navis
import navis.interfaces.neuprint as neu


auth_token = "eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJlbWFpbCI6InZoMjMwOEBueXUuZWR1IiwibGV2ZWwiOiJub2F1dGgiLCJpbWFnZS11cmwiOiJodHRwczovL2xoMy5nb29nbGV1c2VyY29udGVudC5jb20vYS0vQUxWLVVqVzF1TnhSVS1WWHhFWldIVHZIMkhFeXo1cFZCS2RZNUdKdGpqRzd4T19nREtQY3kyWDc4M0s1akJMWHRwU1F2UjYyTGstd3AtTFJfNGpqNDJHSXhVWUYtRURSNVFjVXBKdlZtWV9XNi12b2hTUVAyRnFXVnhxTjdMRDFGNEpYRzdwU3dZRDFFNWFrSHFXN25BaWRmbnh4NE1pQnN2YnQxUU5YYUw2VkVVczRzQnBwMnQ4ZmJmNzJPY2xxY3V6QWVmYTFpcldFOXhHMUVZWWlQSUhfX1VHaF96R0w0aGFNak1iWGdCcUpIcVJsMW5rcVBKenRiaDd3V2lzTWl1czRqZ1JhVlAtU0RyeWpudk9SQ2V1aTFRaW9MbDZvZjB1ZWhEalZZN2hjZXBiVFdzZWktZUJKekxOMHVLVWVtQ2w4Q1NZcEsxSjhuVFdZSFowLS1nUUZOaFFLUHJpalpwc3dwZ1E5TkQ0dFlST0RBRW9GYzMtR1ItTHU1aWRIQjlBd0N2NUxSODNHOW43TU16eWdaS3dnSElQUk1ORl80bkQ4TzM4VV84bnFKeEwxLWxKUk1lYXlXWW05bW1maU5PcXlOZnJyT1IwaDZyVm96RFZmOE9scWRJaG10T1g4blVPeldkQTVVM3E5U1BBRVJMYzJGR1FSRVVtU1E0TFNnY05CZk81cTE5ci04NkxaTWVqd25wNnV1eGhYR2VtNEZsbFBLbk1SZjV2N045alFaVVdLMGo3eHB4YWZwc0dDS0t6SVBMdm5LaWtway1oV2tJUTd4ZFVNN3R6TFR0eG05MFpjcEZJSlY4M1lKTTNNb0Z1ZkNVY3lwLVdFUWttc25najU3dVFQQmI0N1dhTmZqdEd6cEJ5dzZuNjNzSXRkNGR2WjB0eVBoLW5ZeWYtWVpTMDA4VS1KbWg5bFlqT042a0dzdkswQXM2VEIxaFVLZWJ3eVl3aG5wR3ZvUHUyZC0zZDJfTGZMQ05XSXhrQi1VZkQ4ZWxPdld0bXlLaE5XSDZ4OEE5ZEhxQ3RGU1VlVTdHeDdCdERDUHphV0E1TDg4RFFuekF5OHNKSVkwU0kwVk9TNjdzeHQ2WUR1UHBVdFdabHNNUFY3LWJDSm9mUjNKOE1BWGxyMnZ3c0tCa3RSdEd5Yy1uYl9VN0dkejUxRGdxc293dmZvWmh0RXZ3TExLZU9nM3NfRFlNSkhtdWFQU05QUUFLNDlkSEF3dFBHUjQta3BNT0o1ZHB3YXRHcW1qaG1FM1RkWHpzVzh6R1pVTWpqX0I4WG92MEE5MkR4SUZqMzRqR2FlZFV0VUFsZWcxdWduNThhMEZwYnFFQmxxSkl2TGRwN0Q1Mnk2R0N1WDJFcHZJR0JjbGFXVTBXejJRZ2t3REFTc1ZFSm1ScmF2LTFBdGNJblVFVzFFZ0Z0dldmRndKMFBra2cwPXM5Ni1jP3N6PTUwP3N6PTUwIiwiZXhwIjoxOTY0MjMyODUyfQ.2IB-bBQ1iE9PWe2zqD577hzIK2pAf6zCg5ztnbB_UnE"

# client = Client('neuprint.janelia.org', 'manc:v1.2.3', token=auth_token)

client = neu.Client(
    "https://neuprint.janelia.org/",
    token=auth_token,  # use this to instead pass your token directly
    dataset="manc:v1.2.3",
)

# Force PyImageJ to use Conda's Java, ignoring system defaults.
# Must be set BEFORE scyjava/imagej is imported (those trigger JVM startup).
if "CONDA_PREFIX" in os.environ:
    os.environ["JAVA_HOME"] = os.environ["CONDA_PREFIX"]

# Ensure the JVM is started in headless mode from the very first call.
# In a Jupyter kernel the JVM only starts once; if it comes up without the
# right options, the IJ1 legacy layer ends up 'Inactive' and IJ.openImage()
# (which the H5J_Loader_Plugin relies on) silently returns null.
import scyjava
scyjava.config.add_option("-Djava.awt.headless=true")

import imagej

# FIJI_APP = "/home/william-zheng/Downloads/Fiji.app"
# PROJECT_DIR = "/home/william-zheng/Documents/Programming/Python/NeuroInformatics/summer_2026/neuroinfo_fruitfly"
FIJI_APP = "/Users/vuhepola/Desktop/Fiji"
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
FISBe_DIR = DATA_DIR / "FISBe"
FlyLight_DIR = DATA_DIR / "FlyLight"
FANC_DIR = DATA_DIR / "FANC"
MANC_DIR = DATA_DIR / "MANC"

print(FIJI_APP)
print(PROJECT_DIR)
print(DATA_DIR)
print(FISBe_DIR)
print(FlyLight_DIR)
print(FANC_DIR)
print(MANC_DIR)

In [ ]:
client.primary_rois

In [ ]:
roi = 'LegNp(T1)(L)'
target='LegNpT1_L'


# instance=".*T1.*",
class_ = "motor neuron",
regex=True

t1 = neu.fetch_roi(roi)


In [ ]:
criteria = neu.NeuronCriteria(
    rois=roi,
    # target=target,
    class_ = "motor neuron",
    regex=True
    
)


nl, roi_info = neu.fetch_neurons(
    criteria
)
nl

In [ ]:
skels = neu.fetch_skeletons(criteria, with_synapses=True,)

In [ ]:
skels

In [ ]:
skels_um = skels / (1000 / 8)
skels_um

In [ ]:
navis.plot3d(skels_um)

In [ ]:
dps = navis.make_dotprops(skels_um)

In [ ]:
navis.plot3d(dps)

In [ ]:
score_matrix = navis.nblast_allbyall(dps)

In [ ]:
score_matrix_mean = (score_matrix + score_matrix.T)/2

In [ ]:
score_matrix_mean

In [ ]:
def get_neuron_from_dp_id(dp_ids, skels_um, dps):
    
    if type(dp_ids) == np.int64:
        dp_ids = [dp_ids]
    
    neuron_list = []
    for dp_id in dp_ids:
        neuron_list.append(skels_um[skels_um.name == dps[dps.id == dp_id].name.item()][0])
    
    return navis.NeuronList(neuron_list)

def display_matches(score_matrix, ids, skels_um, dps,  k = 1):
    
    if type(ids) == np.int64:
        ids = [ids]
    
    for q_id in ids:
        # print(q_id)
        
        neurons_to_show = []

        m = get_neuron_from_dp_id(
            dp_ids = q_id,
            skels_um=skels_um,
            dps = dps,
            )

        neurons_to_show.extend(m)


        next_best_matches = score_matrix.loc[q_id].nlargest(k+1)
        print(next_best_matches)

        best_ids = next_best_matches.index[1:k+1]
        
        # print(len(best_ids))

        n = get_neuron_from_dp_id(
            dp_ids = best_ids,
            skels_um=skels_um,
            dps = dps,
            )

        # print(len(n))
        neurons_to_show.extend(n)

        # neurons_to_show = navis.NeuronList(neurons_to_show)
        
        navis.plot3d(neurons_to_show)
        
        # print()
        
        
q_ids = score_matrix_mean.index[10]
print(q_ids)

print()
display_matches(
    score_matrix = score_matrix_mean,
    ids = q_ids,
    skels_um = skels_um,
    dps = dps, 
    k = 10
    )